# Taylor Blast Wave: Experiment-Only Notebook

This notebook keeps only the experimental workflow:
- view the explosion photograph sequence,
- calibrate pixels to metres,
- compare measured radii with the best-fit theoretical scaling $r = k t^{2/5}$,
- estimate the explosion energy.

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

img = Image.open('blast.png').convert('RGB')
img_arr = np.array(img)

plt.figure(figsize=(12, 10))
plt.imshow(img_arr)
plt.axis('off')
plt.title('Explosion contact sheet with time stamps and 100 m scale bar')
plt.tight_layout()
plt.show()

print(f'Image size: {img.size[0]} x {img.size[1]} px')

## Calibration

Using the photograph, the 100 m scale bar is approximately **42 pixels** long.

In [ ]:
scale_bar_pixels = 42
metres_per_pixel = 100 / scale_bar_pixels
metres_per_pixel

## Measured Characteristic Radius

These are approximate manual measurements from several frames. We keep two characteristic measurements:
- `radius_px_sides`: horizontal radius from the centreline to the visible side edge
- `radius_px_top`: vertical radius from the base line to the visible top

The theoretical scaling is retained as

$$r = k t^{2/5},$$

with only the constant $k$ fitted from the data.

In [ ]:
# Global measurement toggle used everywhere below.
measurement_mode = 'sides'  # change to 'top' to use the vertical measurement everywhere

In [ ]:
data = pd.DataFrame({
    'time_ms': [15.1, 17.0, 18.8, 20.7, 22.6, 24.4],
    'radius_px_sides': [58, 60, 62, 64, 66, 68],
    'radius_px_top': [40, 41, 42, 43, 44, 45]
})

# Use the shared measurement mode for the scaling plot.
radius_col = f'radius_px_{measurement_mode}'

data['radius_px'] = data[radius_col]
data['time_s'] = data['time_ms'] / 1000
data['radius_m'] = data['radius_px'] * metres_per_pixel

k_fit = np.sum(data['radius_px'] * data['time_ms']**0.4) / np.sum(data['time_ms']**0.8)
tolerance_px = 1.5

data['radius_px_pred'] = k_fit * data['time_ms']**0.4
data['residual_px'] = data['radius_px'] - data['radius_px_pred']
data['abs_error_px'] = np.abs(data['residual_px'])
data['radius_m_pred'] = data['radius_px_pred'] * metres_per_pixel
data['abs_error_m'] = data['radius_m_pred'] - data['radius_m']
data['pct_error'] = 100 * data['abs_error_m'] / data['radius_m']

data

## Best-Fit Theory With Tolerance

In [ ]:
plt.figure(figsize=(7, 5))
plt.scatter(data['time_ms'], data['radius_px'], label='measured radius_px')
plt.plot(data['time_ms'], data['radius_px_pred'], label='best-fit predicted radius_px')
plt.fill_between(
    data['time_ms'],
    data['radius_px_pred'] - tolerance_px,
    data['radius_px_pred'] + tolerance_px,
    alpha=0.2,
    label=f'prediction +/- {tolerance_px:.1f} px tolerance'
)
plt.xlabel('time / ms')
plt.ylabel('radius / px')
plt.title('Measured and best-fit predicted characteristic radius in pixels')
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

In [ ]:
metrics = pd.Series({
    'k_fit_px_per_ms_0p4': k_fit,
    'tolerance_px': tolerance_px,
    'mean_abs_error_px': data['abs_error_px'].mean(),
    'rmse_px': np.sqrt(np.mean(data['residual_px']**2)),
    'mean_abs_percent_error': np.mean(np.abs(data['pct_error'])),
    'max_abs_percent_error': np.max(np.abs(data['pct_error'])),
    'points_within_tolerance': int((data['abs_error_px'] <= tolerance_px).sum()),
    'total_points': int(len(data)),
    'fraction_within_tolerance': (data['abs_error_px'] <= tolerance_px).mean(),
})
metrics

In [ ]:
plt.figure(figsize=(7, 4))
plt.axhline(0, color='black', linewidth=1)
plt.axhline(tolerance_px, color='tab:red', linestyle='--', label=f'+{tolerance_px:.1f} px tolerance')
plt.axhline(-tolerance_px, color='tab:red', linestyle='--', label=f'-{tolerance_px:.1f} px tolerance')
plt.scatter(data['time_ms'], data['residual_px'], color='tab:purple', label='pixel residual')
plt.plot(data['time_ms'], data['residual_px'], color='tab:purple', alpha=0.6)
plt.xlabel('time / ms')
plt.ylabel('residual / px')
plt.title('Residuals: measured radius_px minus best-fit prediction')
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

## Explosion Energy Estimate

Using the order-of-magnitude estimate

$$E \sim \frac{\rho r^5}{t^2},$$

with air density $\rho \approx 1.2\ \mathrm{kg\,m^{-3}}$.

The energy estimate below uses the same `measurement_mode` toggle as the scaling section above.

In [ ]:
energy_radius_col = f'radius_px_{measurement_mode}'

data['energy_radius_px'] = data[energy_radius_col]
data['energy_radius_m'] = data['energy_radius_px'] * metres_per_pixel

print(f'Using {energy_radius_col} for the energy estimate')

In [ ]:
rho = 1.2

data['E_est_J'] = rho * data['energy_radius_m']**5 / data['time_s']**2
data['E_est_kilotons_TNT'] = data['E_est_J'] / 4.184e12

energy_summary = data[[
    'time_ms', 'energy_radius_px', 'energy_radius_m', 'E_est_J', 'E_est_kilotons_TNT'
]].copy()
energy_summary.columns = ['time_ms', 'radius_px_used', 'radius_m_used', 'E_est_J', 'E_est_kilotons_TNT']
energy_summary

In [ ]:
E_mean = data['E_est_J'].mean()
kt_mean = E_mean / 4.184e12

print(f'Mean estimated energy: {E_mean:.3e} J')
print(f'Mean estimated yield: {kt_mean:.2f} kilotons TNT')